# Facial Emotion Recognition — CNN Pipeline

Trains and evaluates every CNN model in this repo end to end: the VGG-style
baseline, the improved GAP/augmentation/label-smoothing CNN, the ResNet-SE
+MixUp+EMA model, and the gated landmark-fusion model (+ its 4-way ablation).
Each cell runs one script from `CNN/` and displays its outputs inline, the
same pattern `SVM_notebook.ipynb` uses for the classical pipeline.

For the full write-up of what each model changed and why, plus per-class
numbers and confusion matrices, see the root [`README.md`](README.md#results)
and report.

## Why four models

Each CNN targets the previous one's specific failure mode:

| Model | Key changes | Why |
|---|---|---|
| Baseline VGG | 3 conv blocks + big FC head | Standard FER baseline; overfits (train 0.74 / val 0.66), weakest on `fear` and `sad` |
| GAP + Aug + LS | Global Average Pooling head, +1 block, stronger aug, label smoothing, cosine LR | Removes the ~2.4M-param FC (main overfit source) |
| ResNet-SE + MixUp + EMA | Residual blocks + Squeeze-Excitation, MixUp/CutMix, EMA, more epochs | Depth + attention + strongest regularizer, once the earlier gains plateaued |
| Gated landmark fusion | Fine-tunes the GAP backbone with a learned gate onto 26 dlib landmark features | Tests whether handcrafted geometry adds signal on top of a CNN — the project's central question |

## Full run vs. smoke test

Full training here takes **hours** — the ResNet-SE model alone
takes ~3.7h on a Colab GPU. Set `FULL_RUN` in the first code cell:

- `FULL_RUN = False` (default): every model trains for **1 epoch**, writing
  to sibling `checkpoints_smoketest/` / `results_smoketest/` folders — just
  to confirm the code path runs end to end.
- `FULL_RUN = True`: every model trains for its real default number of
  epochs (40–60), writing to the real `checkpoints/` / `results/` —
  reproducing (and overwriting) the actual reported results.

Results section always reads the real `results/`
folders, not smoke test ones, so we always get the
real, reported numbers there regardless of `FULL_RUN`.

When doing a long unattended `FULL_RUN=True` pass, wrap the whole
`jupyter nbconvert --execute` invocation in
`caffeinate -i`.

## Caching (`REBUILD`)

The second code cell also has a `REBUILD` toggle:

- `REBUILD = False` (default): each Part **reuses** the outputs it already
  produced — a trained checkpoint a `metrics.json`.
- `REBUILD = True`: forces every step to recompute from scratch for the
  current `FULL_RUN` mode, overwriting what's on disk.

The dlib landmark extraction (`build_alignment.py`) is always
governed by the cache — it never
re-extracts (delete the cache to force it).

## Setup

Everything in this notebook runs in the project's `appliedml` conda env
(sklearn, cvxopt, torch, torchvision):

```bash
conda env create -f environment_appliedml.yml   # once
conda activate appliedml
```

Run this notebook from the repo root and pick the `appliedml` kernel. Note:
the dlib landmark extraction in Part 4 (`build_alignment.py`) needs the
separate feature-extraction env (`fer_feature_extraction`, from
`environment.yml`); everything else here runs under `appliedml`.

In [ ]:
import os
assert os.path.exists("paths.py"), (
    "paths.py not found. Make sure the notebook is opened from the "
    "emotion-recognition/ root directory.")
print("Working directory:", os.getcwd())
print("paths.py found")

# For full run, change this to true
FULL_RUN = False 

# Cache behaviour. When False (default), every Part reuses the outputs it
# already produced. Set True to force every step to recompute from scratch,
# overwriting whatever is on disk for the current FULL_RUN mode.
REBUILD = False

In [ ]:
import subprocess, sys, json, os

def run(module, *args):
    """Run `python -m <module> <args>` from the repo root, streaming output."""
    cmd = [sys.executable, "-m", module, *args]
    print(">", " ".join(cmd))
    result = subprocess.run(cmd)
    print("Return code:", result.returncode)
    return result.returncode

def cached_run(outputs, module, *args):
    """Run `module` unless its outputs are already cached.

    `outputs` is a path or list of paths this step produces. If they all
    already exist and REBUILD is False, the step is skipped and its cached
    outputs are reused. Set REBUILD=True (cell 1) to force a recompute.
    Returns 0 on a cache hit (same as a clean run).
    """
    paths = [outputs] if isinstance(outputs, str) else list(outputs)
    if not REBUILD and all(os.path.exists(p) for p in paths):
        print(f"[cache] reusing {', '.join(paths)} — skipping {module} "
              f"(set REBUILD=True to force)")
        return 0
    return run(module, *args)

def epoch_args():
    """1 epoch in smoke-test mode, the script's real default otherwise."""
    return [] if FULL_RUN else ["--epochs", "1"]

def io_dirs(model_dir):
    """(ckpt_dir, results_dir) for a model, isolated per FULL_RUN.

    FULL_RUN=True writes to the real checkpoints/ and results/ — the ones
    RESULTS.md and README.md quote. FULL_RUN=False (smoke test) writes to
    sibling *_smoketest/ dirs instead, so a 1-epoch sanity check can never
    overwrite the real, documented checkpoints or metrics.
    """
    suffix = "" if FULL_RUN else "_smoketest"
    return (f"{model_dir}/checkpoints{suffix}", f"{model_dir}/results{suffix}")

---
# Part 1 — Baseline CNN

**Scripts:** `CNN/baseline_CNN/train.py`, `CNN/baseline_CNN/test.py`
**Output:** `CNN/baseline_CNN/checkpoints/`, `CNN/baseline_CNN/results/`

VGG-style CNN (3 conv blocks 64→128→256 + dropout FC head) trained directly
on 48×48 grayscale FER-2013 images, no landmark features. Standard FER
baseline — it overfits (train acc 0.740 vs val 0.666) and is weakest on
`fear` (F1 0.47) and `sad` (0.54); every later model targets that gap. See the root
[`README.md`](README.md#cnn-progression) for the full design rationale.

In [ ]:
_ckpt_dir, _results_dir = io_dirs("CNN/baseline_CNN")
cached_run(f"{_ckpt_dir}/best.pt",
           "CNN.baseline_CNN.train", "--ckpt-dir", _ckpt_dir, "--results-dir", _results_dir, *epoch_args())

In [ ]:
cached_run(f"{_results_dir}/metrics.json",
           "CNN.baseline_CNN.test", "--ckpt", f"{_ckpt_dir}/best.pt", "--results-dir", _results_dir)

---
# Part 2 — Improved CNN (GAP + augmentation + label smoothing)

**Scripts:** `CNN/cnn_gap_aug_labelsmooth/train.py`, `.../test.py`
**Output:** `CNN/cnn_gap_aug_labelsmooth/checkpoints/`, `.../results/`

Same dataset, same seeded 10% val split, same FLOPs accounting as the
baseline — only the model and recipe change. Swaps the baseline's ~2.4M-param
FC head for Global Average Pooling (the main overfitting fix), adds a 4th
conv block, stronger augmentation, label smoothing, and cosine LR + warmup.
Also serves as the pretrained backbone for the fusion model in Part 4.
Evaluated with test-time augmentation (`--tta`). Result: +3.1 accuracy points
/ +1.4 macro-F1 points over the baseline, with `fear` (the baseline's
weakest class) improving the most (+0.076 F1). See the root
[`README.md`](README.md#cnn-progression) for the full change table.

In [ ]:
_ckpt_dir, _results_dir = io_dirs("CNN/cnn_gap_aug_labelsmooth")
cached_run(f"{_ckpt_dir}/best.pt",
           "CNN.cnn_gap_aug_labelsmooth.train", "--ckpt-dir", _ckpt_dir, "--results-dir", _results_dir, *epoch_args())

In [ ]:
cached_run(f"{_results_dir}/metrics.json",
           "CNN.cnn_gap_aug_labelsmooth.test", "--tta", "--ckpt", f"{_ckpt_dir}/best.pt", "--results-dir", _results_dir)

---
# Part 3 — ResNet-SE + MixUp + EMA

**Scripts:** `CNN/cnn_resnet_se_mixup_ema/train.py`, `.../test.py`
**Output:** `CNN/cnn_resnet_se_mixup_ema/checkpoints/`, `.../results/`

The strongest single CNN in this repo. The model plateaued around
val ≈0.69 with the train/val gap mostly closed by augmentation + label
smoothing — more headroom needs both more capacity and a
stronger regularizer. This model adds pre-activation residual blocks (≈18
layers deep) with Squeeze-and-Excitation channel attention, MixUp/CutMix
sample mixing, and an EMA weight average, trained for 60 epochs (vs. 40 for
the other two). Evaluated with multi-crop TTA (`--tta-multi`).

In [ ]:
_ckpt_dir, _results_dir = io_dirs("CNN/cnn_resnet_se_mixup_ema")
cached_run(f"{_ckpt_dir}/best.pt",
           "CNN.cnn_resnet_se_mixup_ema.train", "--ckpt-dir", _ckpt_dir, "--results-dir", _results_dir, *epoch_args())

In [ ]:
cached_run(f"{_results_dir}/metrics.json",
           "CNN.cnn_resnet_se_mixup_ema.test", "--tta-multi", "--ckpt", f"{_ckpt_dir}/best.pt", "--results-dir", _results_dir)

---
# Part 4 — Gated landmark-fusion model + 4-way ablation

**Scripts:** `CNN/fusion_gated/build_alignment.py`, `train.py`, `test.py`
**Output:** `CNN/fusion_gated/cache/`, `checkpoints/`, `results/`

Fine-tunes the Part 2 backbone with a learned, gated residual connection to
the 26 precomputed dlib landmark features from the SVM side — the project's
central test of whether handcrafted geometry adds signal on top of a CNN.
`z_fused = z_cnn + tanh(alpha) * mask * g(landmarks)`, with `alpha` starting
at 0 so the model begins as a pure CNN and only grows the gate if geometry
genuinely helps. See the root [`README.md`](README.md#fusion-architecture) for the architecture
diagram and the two documented deviations from the original spec.

Three variants are trained for the ablation: the default gated model, a
concat-fusion variant, and a frozen-gate (`alpha=1`) variant. 

As described in the next part, the gate barely opens (α_final ≈ 0.055) and none of the fusion
variants meaningfully beat the CNN-only backbone — geometry adds no reliable
signal once the CNN already sees the raw pixels.

In [ ]:
_align_cache = ["CNN/fusion_gated/cache/align_train.npz",
                "CNN/fusion_gated/cache/align_test.npz",
                "CNN/fusion_gated/cache/scaler.npz"]
if all(os.path.exists(p) for p in _align_cache):
    print(f"[cache] reusing {', '.join(_align_cache)} — skipping build_alignment")
else:
    run("CNN.fusion_gated.build_alignment")

In [ ]:
_ckpt_dir, _results_dir = io_dirs("CNN/fusion_gated")
cached_run(f"{_ckpt_dir}/best.pt",
           "CNN.fusion_gated.train", "--finetune", "--ckpt-dir", _ckpt_dir, "--results-dir", _results_dir, *epoch_args())
cached_run(f"{_results_dir}/metrics_gated.json",
           "CNN.fusion_gated.test", "--ckpt", f"{_ckpt_dir}/best.pt", "--results-dir", _results_dir)

In [ ]:
cached_run(f"{_ckpt_dir}/best_concat.pt",
           "CNN.fusion_gated.train", "--finetune", "--ablation", "concat", "--ckpt-dir", _ckpt_dir, "--results-dir", _results_dir, *epoch_args())
cached_run(f"{_results_dir}/metrics_concat.json",
           "CNN.fusion_gated.test", "--ckpt", f"{_ckpt_dir}/best_concat.pt", "--results-dir", _results_dir)

In [ ]:
cached_run(f"{_ckpt_dir}/best_frozen.pt",
           "CNN.fusion_gated.train", "--finetune", "--ablation", "frozen", "--ckpt-dir", _ckpt_dir, "--results-dir", _results_dir, *epoch_args())
cached_run(f"{_results_dir}/metrics_frozen.json",
           "CNN.fusion_gated.test", "--ckpt", f"{_ckpt_dir}/best_frozen.pt", "--results-dir", _results_dir)

---
# Part 5 — Combined results

This cell summarizes the documented test-set numbers so the notebook reads
standalone without executing anything below — the code cells that follow
regenerate the same table, confusion matrices, and curves live from each
`CNN/*/results/` folder. Macro-F1 is the primary metric because of the severe class imbalance,
disgust ≈ 1.5% vs happy ≈ 25%.)*

| Model | Accuracy | Macro-F1 | Weighted-F1 |
|---|---|---|---|
| Baseline VGG CNN | 0.665 | 0.647 | 0.663 |
| GAP + Aug + LS CNN (TTA) | 0.696 | 0.661 | 0.696 |
| **ResNet-SE + MixUp + EMA CNN (TTA)** | **0.697** | **0.670** | **0.695** |
| CNN + concat fusion | 0.702 | 0.682 | 0.701 |
| CNN + gated fusion (α init 0) | 0.680 | 0.638 | 0.682 |

**Validation-accuracy progression** across the three standalone CNNs (each
targets the previous model's failure mode):

![CNN validation-accuracy progression](figures/cnn_val_acc_progression.png)

**Confusion matrices — baseline vs. best model** (row-normalized). Every
diagonal strengthens except the near-ceiling `happy`/`surprise`; `fear`→`sad`
and `sad`→`neutral` stay the dominant residual confusions:

| Baseline CNN | ResNet-SE (best) |
|---|---|
| ![Baseline confusion matrix](figures/cnn_baseline_cm.png) | ![Best confusion matrix](figures/cnn_best_cm.png) |

**Fusion — the project's central test.** The 26 dlib landmark features are
injected into the GAP-CNN through a learned gate `tanh(α)` initialized at 0,
so the model *starts* as a pure CNN and only opens the gate if geometry helps.
It doesn't: the gate flatlines near zero (final ≈ 0.055) and gated fusion
*underperforms* CNN-only. At 48×48 and ~80% detection, hand-crafted geometry
adds no reliable signal on top of a well-regularized CNN — one of the
project's main findings.

![Fusion gate stays near zero](figures/fusion_alpha.png)

The full write-up with the compute analysis is in the root
[`README.md`](README.md#results) and the report


In [ ]:
import json, os

MODELS = {
    "Baseline CNN":            "CNN/baseline_CNN/results/metrics.json",
    "Improved CNN (GAP+aug)":  "CNN/cnn_gap_aug_labelsmooth/results/metrics.json",
    "ResNet-SE+MixUp+EMA":     "CNN/cnn_resnet_se_mixup_ema/results/metrics.json",
    "Fusion (gated)":          "CNN/fusion_gated/results/metrics_gated.json",
    "Fusion (concat)":         "CNN/fusion_gated/results/metrics_concat.json",
    "Fusion (frozen gate)":    "CNN/fusion_gated/results/metrics_frozen.json",
}

rows = []
for name, path in MODELS.items():
    if not os.path.exists(path):
        rows.append((name, None, None, None))
        continue
    with open(path) as f:
        m = json.load(f)
    rows.append((name, m["accuracy"], m["macro_f1"], m["weighted_f1"]))

header = f'{"Model":<26} {"Accuracy":>10} {"Macro F1":>10} {"Weighted F1":>12}'
print(header)
print("-" * len(header))
for name, acc, mf1, wf1 in rows:
    if acc is None:
        print(f"{name:<26} {'— (not run) —':>34}")
    else:
        print(f"{name:<26} {acc:>10.4f} {mf1:>10.4f} {wf1:>12.4f}")


In [ ]:
import matplotlib.pyplot as plt
from CNN.visualize import plot_confusion_matrix
import numpy as np

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for ax, (name, path) in zip(axes.flat, MODELS.items()):
    if not os.path.exists(path):
        ax.axis("off")
        ax.set_title(f"{name}\n(not found — run training cell above)")
        continue
    with open(path) as f:
        m = json.load(f)
    cm = np.array(m["confusion_matrix"])
    classes = m["classes"]
    y_true, y_pred = [], []
    for i, row in enumerate(cm):
        for j, count in enumerate(row):
            y_true += [i] * int(count)
            y_pred += [j] * int(count)
    plot_confusion_matrix(y_true, y_pred, classes, title=name, ax=ax)

plt.tight_layout()
plt.show()


### Learning curves

In [ ]:
from CNN.visualize import plot_learning_curve

HISTORIES = {
    "Baseline CNN":           "CNN/baseline_CNN/results/history.json",
    "Improved CNN (GAP+aug)": "CNN/cnn_gap_aug_labelsmooth/results/history.json",
    "ResNet-SE+MixUp+EMA":    "CNN/cnn_resnet_se_mixup_ema/results/history.json",
    "Fusion (gated)":         "CNN/fusion_gated/results/history_gated.json",
}

for name, path in HISTORIES.items():
    if not os.path.exists(path):
        print(f"Not found: {path} — run the training cell above first.")
        continue
    with open(path) as f:
        history = json.load(f)
    plot_learning_curve(history, title=name)
